In [ ]:
import os
import glob
import json
import zipfile
import shutil
import librosa
import soundfile as sf
import numpy as np
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive', force_remount=True)

# 2. 경로 설정
BASE_DIR = '/content/drive/MyDrive/UNet'
NOISE_DIR = f'{BASE_DIR}/Noise'
OUTPUT_DIR = f'{BASE_DIR}/Clean_Noise_Zips'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# 임시 작업 폴더
TEMP_TS_DIR = '/content/temp_ts'
TEMP_TL_DIR = '/content/temp_tl'
TEMP_CLEAN_DIR = '/content/temp_clean'

def process_noise_datasets():
    # 폴더 초기화
    for d in [TEMP_TS_DIR, TEMP_TL_DIR, TEMP_CLEAN_DIR]:
        if os.path.exists(d): shutil.rmtree(d)
        os.makedirs(d)

    # TS 압축 파일 목록
    ts_zips = sorted(glob.glob(f'{NOISE_DIR}/TS_*.zip'))
    total_processed = 0

    for ts_zip_path in ts_zips:
        filename = os.path.basename(ts_zip_path)
        tl_zip_path = os.path.join(NOISE_DIR, filename.replace('TS_', 'TL_'))

        if not os.path.exists(tl_zip_path):
            print(f"⚠️ 매칭되는 라벨 파일 없음: {tl_zip_path}")
            continue

        print(f"\n📦 압축 해제 중: {filename} & {os.path.basename(tl_zip_path)}")

        # TS와 TL 각각 압축 해제
        with zipfile.ZipFile(ts_zip_path, 'r') as z: z.extractall(TEMP_TS_DIR)
        with zipfile.ZipFile(tl_zip_path, 'r') as z: z.extractall(TEMP_TL_DIR)

        # 압축 해제된 라벨(JSON) 파일들을 파일명 기준으로 매핑 (폴더 구조 불일치 방지)
        json_files = glob.glob(f'{TEMP_TL_DIR}/**/*.json', recursive=True)
        label_map = {os.path.splitext(os.path.basename(p))[0]: p for p in json_files}

        audio_files = glob.glob(f'{TEMP_TS_DIR}/**/*.mp3', recursive=True)
        print(f"  -> {len(audio_files)}개의 오디오 파일 전처리 시작...")

        for audio_path in audio_files:
            file_id = os.path.splitext(os.path.basename(audio_path))[0]
            label_path = label_map.get(file_id)

            if not label_path:
                continue

            try:
                # JSON 파싱하여 유효 구간(Segmentations) 추출
                with open(label_path, 'r', encoding='utf-8') as f:
                    label_data = json.load(f)

                segmentations = label_data.get('LabelDataInfo', {}).get('Segmentations', [])
                if not segmentations:
                    continue

                y, sr = librosa.load(audio_path, sr=16000)
                clean_segments = []

                # 해당 오디오의 모든 유효 구간을 잘라서 배열에 담기
                for seg in segmentations:
                    start_sec, end_sec = seg[0], seg[1]
                    start_idx = int(start_sec * sr)
                    end_idx = int(end_sec * sr)

                    if start_idx < len(y):
                        clean_segments.append(y[start_idx:min(end_idx, len(y))])

                if not clean_segments:
                    continue

                # 유효 구간들만 이어 붙이기
                final_audio = np.concatenate(clean_segments)

                # 원본의 디렉토리 구조를 반영하여 WAV로 저장
                rel_path = os.path.relpath(audio_path, TEMP_TS_DIR)
                save_path = os.path.join(TEMP_CLEAN_DIR, rel_path).replace('.mp3', '.wav')
                os.makedirs(os.path.dirname(save_path), exist_ok=True)
                sf.write(save_path, final_audio, sr)

                total_processed += 1

            except Exception as e:
                pass # 에러가 발생하는 손상된 파일은 무시

        # 다음 Zip 파일을 위해 임시 폴더 비우기
        shutil.rmtree(TEMP_TS_DIR); os.makedirs(TEMP_TS_DIR)
        shutil.rmtree(TEMP_TL_DIR); os.makedirs(TEMP_TL_DIR)

    # 전처리된 모든 오디오를 단일 파일로 묶기
    out_zip_path = os.path.join(OUTPUT_DIR, 'Cleaned_Noise_All.zip')
    print(f"\n📦 모든 전처리 완료({total_processed}개). 단일 파일로 압축 중: {out_zip_path}")

    with zipfile.ZipFile(out_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(TEMP_CLEAN_DIR):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, TEMP_CLEAN_DIR)
                zipf.write(file_path, arcname)

    print("✅ 전처리 및 재압축 완료.")

process_noise_datasets()

Mounted at /content/drive

📦 압축 해제 중: TS_1.자연.zip & TL_1.자연.zip
  -> 2843개의 오디오 파일 전처리 시작...

📦 압축 해제 중: TS_10.기계 및 공구.zip & TL_10.기계 및 공구.zip
  -> 3427개의 오디오 파일 전처리 시작...

📦 압축 해제 중: TS_2.무기.zip & TL_2.무기.zip
  -> 3810개의 오디오 파일 전처리 시작...

📦 압축 해제 중: TS_3.사람.zip & TL_3.사람.zip
  -> 2667개의 오디오 파일 전처리 시작...

📦 압축 해제 중: TS_4.동물.zip & TL_4.동물.zip
  -> 3113개의 오디오 파일 전처리 시작...

📦 압축 해제 중: TS_5.알람.zip & TL_5.알람.zip
  -> 1003개의 오디오 파일 전처리 시작...

📦 압축 해제 중: TS_6.물체.zip & TL_6.물체.zip
  -> 8655개의 오디오 파일 전처리 시작...

📦 압축 해제 중: TS_7.악기.zip & TL_7.악기.zip
  -> 3134개의 오디오 파일 전처리 시작...

📦 압축 해제 중: TS_8.군부대 운송수단.zip & TL_8.군부대 운송수단.zip
  -> 2995개의 오디오 파일 전처리 시작...

📦 압축 해제 중: TS_9.생활.zip & TL_9.생활.zip
  -> 4201개의 오디오 파일 전처리 시작...

📦 모든 전처리 완료(35848개). 단일 파일로 압축 중: /content/drive/MyDrive/UNet/Clean_Noise_Zips/Cleaned_Noise_All.zip
✅ 전처리 및 재압축 완료.


In [ ]:
import os
from google.colab import drive

check_path = '/content/drive/MyDrive/UNet/Clean_Noise_Zips/Cleaned_Noise_All.zip'

# 1. 파일 생성 여부 및 용량 확인
if os.path.exists(check_path):
    file_size_mb = os.path.getsize(check_path) / (1024 * 1024)
    print(f"파일이 정상적으로 생성되었습니다. (용량: {file_size_mb:.2f} MB)")
else:
    print("파일을 찾을 수 없습니다.")

# 2. 구글 드라이브 강제 동기화 (메모리에 남은 데이터를 드라이브로 쓰기)
drive.flush_and_unmount()
print("드라이브 동기화가 요청되었습니다. 구글 드라이브 웹페이지를 새로고침 해보세요.")

파일이 정상적으로 생성되었습니다. (용량: 13944.53 MB)
드라이브 동기화가 요청되었습니다. 구글 드라이브 웹페이지를 새로고침 해보세요.


In [ ]:
import os
import glob
import json
import zipfile
import shutil
import librosa
import soundfile as sf
import numpy as np
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive', force_remount=True)

# 2. 경로 설정
BASE_DIR = '/content/drive/MyDrive/UNet'
NOISE_DIR = f'{BASE_DIR}/Noise'
OUTPUT_DIR = f'{BASE_DIR}/Clean_Noise_Zips'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# 임시 작업 폴더
TEMP_TS_DIR = '/content/temp_vs'
TEMP_TL_DIR = '/content/temp_vl'
TEMP_CLEAN_DIR = '/content/temp_clean_v'

def process_noise_datasets():
    # 폴더 초기화
    for d in [TEMP_TS_DIR, TEMP_TL_DIR, TEMP_CLEAN_DIR]:
        if os.path.exists(d): shutil.rmtree(d)
        os.makedirs(d)

    # TS 압축 파일 목록
    ts_zips = sorted(glob.glob(f'{NOISE_DIR}/VS_*.zip'))
    total_processed = 0

    for ts_zip_path in ts_zips:
        filename = os.path.basename(ts_zip_path)
        tl_zip_path = os.path.join(NOISE_DIR, filename.replace('VS_', 'VL_'))

        if not os.path.exists(tl_zip_path):
            print(f"⚠️ 매칭되는 라벨 파일 없음: {tl_zip_path}")
            continue

        print(f"\n📦 압축 해제 중: {filename} & {os.path.basename(tl_zip_path)}")

        # TS와 TL 각각 압축 해제
        with zipfile.ZipFile(ts_zip_path, 'r') as z: z.extractall(TEMP_TS_DIR)
        with zipfile.ZipFile(tl_zip_path, 'r') as z: z.extractall(TEMP_TL_DIR)

        # 압축 해제된 라벨(JSON) 파일들을 파일명 기준으로 매핑 (폴더 구조 불일치 방지)
        json_files = glob.glob(f'{TEMP_TL_DIR}/**/*.json', recursive=True)
        label_map = {os.path.splitext(os.path.basename(p))[0]: p for p in json_files}

        audio_files = glob.glob(f'{TEMP_TS_DIR}/**/*.mp3', recursive=True)
        print(f"  -> {len(audio_files)}개의 오디오 파일 전처리 시작...")

        for audio_path in audio_files:
            file_id = os.path.splitext(os.path.basename(audio_path))[0]
            label_path = label_map.get(file_id)

            if not label_path:
                continue

            try:
                # JSON 파싱하여 유효 구간(Segmentations) 추출
                with open(label_path, 'r', encoding='utf-8') as f:
                    label_data = json.load(f)

                segmentations = label_data.get('LabelDataInfo', {}).get('Segmentations', [])
                if not segmentations:
                    continue

                y, sr = librosa.load(audio_path, sr=16000)
                clean_segments = []

                # 해당 오디오의 모든 유효 구간을 잘라서 배열에 담기
                for seg in segmentations:
                    start_sec, end_sec = seg[0], seg[1]
                    start_idx = int(start_sec * sr)
                    end_idx = int(end_sec * sr)

                    if start_idx < len(y):
                        clean_segments.append(y[start_idx:min(end_idx, len(y))])

                if not clean_segments:
                    continue

                # 유효 구간들만 이어 붙이기
                final_audio = np.concatenate(clean_segments)

                # 원본의 디렉토리 구조를 반영하여 WAV로 저장
                rel_path = os.path.relpath(audio_path, TEMP_TS_DIR)
                save_path = os.path.join(TEMP_CLEAN_DIR, rel_path).replace('.mp3', '.wav')
                os.makedirs(os.path.dirname(save_path), exist_ok=True)
                sf.write(save_path, final_audio, sr)

                total_processed += 1

            except Exception as e:
                pass # 에러가 발생하는 손상된 파일은 무시

        # 다음 Zip 파일을 위해 임시 폴더 비우기
        shutil.rmtree(TEMP_TS_DIR); os.makedirs(TEMP_TS_DIR)
        shutil.rmtree(TEMP_TL_DIR); os.makedirs(TEMP_TL_DIR)

    # 전처리된 모든 오디오를 단일 파일로 묶기
    out_zip_path = os.path.join(OUTPUT_DIR, 'Cleaned_Noise_All_Val.zip')
    print(f"\n📦 모든 전처리 완료({total_processed}개). 단일 파일로 압축 중: {out_zip_path}")

    with zipfile.ZipFile(out_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(TEMP_CLEAN_DIR):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, TEMP_CLEAN_DIR)
                zipf.write(file_path, arcname)

    print("✅ 전처리 및 재압축 완료.")

process_noise_datasets()

Mounted at /content/drive

📦 압축 해제 중: VS_1.자연.zip & VL_1.자연.zip
  -> 355개의 오디오 파일 전처리 시작...

📦 압축 해제 중: VS_10.기계 및 공구.zip & VL_10.기계 및 공구.zip
  -> 429개의 오디오 파일 전처리 시작...

📦 압축 해제 중: VS_2.무기.zip & VL_2.무기.zip
  -> 472개의 오디오 파일 전처리 시작...

📦 압축 해제 중: VS_3.사람.zip & VL_3.사람.zip
  -> 338개의 오디오 파일 전처리 시작...

📦 압축 해제 중: VS_4.동물.zip & VL_4.동물.zip
  -> 390개의 오디오 파일 전처리 시작...

📦 압축 해제 중: VS_5.알람.zip & VL_5.알람.zip
  -> 128개의 오디오 파일 전처리 시작...

📦 압축 해제 중: VS_6.물체.zip & VL_6.물체.zip
  -> 1070개의 오디오 파일 전처리 시작...

📦 압축 해제 중: VS_7.악기.zip & VL_7.악기.zip
  -> 399개의 오디오 파일 전처리 시작...

📦 압축 해제 중: VS_8.군부대 운송수단.zip & VL_8.군부대 운송수단.zip
  -> 374개의 오디오 파일 전처리 시작...

📦 압축 해제 중: VS_9.생활.zip & VL_9.생활.zip
  -> 526개의 오디오 파일 전처리 시작...

📦 모든 전처리 완료(4481개). 단일 파일로 압축 중: /content/drive/MyDrive/UNet/Clean_Noise_Zips/Cleaned_Noise_All_Val.zip
✅ 전처리 및 재압축 완료.


In [ ]:
import os
from google.colab import drive

check_path = '/content/drive/MyDrive/UNet/Clean_Noise_Zips/Cleaned_Noise_All_Val.zip'

# 1. 파일 생성 여부 및 용량 확인
if os.path.exists(check_path):
    file_size_mb = os.path.getsize(check_path) / (1024 * 1024)
    print(f"파일이 정상적으로 생성되었습니다. (용량: {file_size_mb:.2f} MB)")
else:
    print("파일을 찾을 수 없습니다.")

# 2. 구글 드라이브 강제 동기화 (메모리에 남은 데이터를 드라이브로 쓰기)
drive.flush_and_unmount()
print("드라이브 동기화가 요청되었습니다. 구글 드라이브 웹페이지를 새로고침 해보세요.")

파일이 정상적으로 생성되었습니다. (용량: 1684.41 MB)
드라이브 동기화가 요청되었습니다. 구글 드라이브 웹페이지를 새로고침 해보세요.
